# Procesar CSV con Python y PySpark

Usaremos el mismo CSV y el mismo cálculo en ambos casos. Completa antes la [guía de entorno de PySpark](../01_environment_pyspark.md).

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from time import perf_counter
import csv

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[2]").appName("Comparar CSV").getOrCreate()
carpeta = Path(mkdtemp(prefix="pyspark_csv_"))
print("Archivos de la práctica:", carpeta)

## 1. Preparar el CSV

Primero mira una tabla pequeña y su texto. Después genera un millón de pedidos ficticios. Puedes reducir la cifra a 100 000 si la VM tiene poco espacio.

In [ ]:
pequeno = spark.createDataFrame(
    [(1, "Pan", 2), (2, "Manzanas", 4), (3, "Pan", 1)],
    ["pedido", "producto", "unidades"],
)
csv_pequeno = carpeta / "pequeno_csv"
pequeno.coalesce(1).write.csv(str(csv_pequeno), header=True)
print(next(csv_pequeno.glob("part-*.csv")).read_text(encoding="utf-8"))

In [ ]:
filas = 1_000_000
grande = spark.range(filas, numPartitions=4).select(
    F.col("id").alias("pedido"),
    F.concat(F.lit("zona_"), (F.col("id") % 20)).alias("zona"),
    (F.col("id") % 12 + 1).alias("unidades"),
    F.sha2(F.col("id").cast("string"), 256).alias("referencia"),
)

csv_grande = carpeta / f"pedidos_{filas}_csv"
grande.write.mode("overwrite").csv(str(csv_grande), header=True)
print("Tamaño en MiB:", sum(f.stat().st_size for f in csv_grande.glob("part-*.csv")) / 1024**2)

## 2. Comparar la misma consulta

Suma las unidades de la zona 3. Python lee una fila cada vez con su módulo csv. PySpark reparte el trabajo en tareas. Repite para observar cuánto influyen el arranque y la caché del sistema.

In [ ]:
def sumar_con_python(ruta):
    total = 0
    for parte in ruta.glob("part-*.csv"):
        with parte.open(encoding="utf-8", newline="") as archivo:
            for fila in csv.DictReader(archivo):
                if fila["zona"] == "zona_3":
                    total += int(fila["unidades"])
    return total

def consulta(tabla):
    return tabla.filter(F.col("zona") == "zona_3").agg(F.sum("unidades"))

tiempos = {}

def medir(nombre, calcular):
    inicio = perf_counter()
    total = calcular()
    tiempos[nombre] = perf_counter() - inicio
    print(f"{nombre}: {total} unidades en {tiempos[nombre]:.2f} s")
    return total

esquema = "pedido LONG, zona STRING, unidades LONG, referencia STRING"
total_python = medir("Python con CSV", lambda: sumar_con_python(csv_grande))
total_csv = medir("Spark con CSV", lambda: consulta(
    spark.read.csv(str(csv_grande), header=True, schema=esquema)
).first()[0])
assert total_python == total_csv


## 3. Resultado de la comparación

Ambos leen el mismo CSV y suman las unidades de la zona 3. Con un millón de filas, ambos deben obtener **400 000 unidades**. Python recorre las filas una por una. PySpark reparte el trabajo en tareas. Los tiempos indican qué ocurrió en esta ejecución de la VM y pueden cambiar al repetirla.

In [ ]:
print(f"Python:  {total_python:,} unidades | {tiempos['Python con CSV']:.3f} s")
print(f"PySpark: {total_csv:,} unidades | {tiempos['Spark con CSV']:.3f} s")

diferencia = tiempos["Python con CSV"] - tiempos["Spark con CSV"]
if abs(diferencia) < 0.001:
    print("Los tiempos medidos son prácticamente iguales.")
else:
    mas_rapido = "PySpark" if diferencia > 0 else "Python"
    print(f"En esta ejecución, {mas_rapido} tardó {abs(diferencia):.3f} s menos.")

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Siguiente: comparar Parquet](06_parquet_python_vs_pyspark.ipynb)